In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
import string
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

input_csv = "/content/drive/MyDrive/Thesis/Code/text_classifier/data/df_concat.csv"
df = pd.read_csv(input_csv )
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
def clean_text(text):

    # Remove URLs https://www.
    ulr_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = ulr_pattern.sub(r'', text)

    # Remove HTML tags
    html_pattern = re.compile('<.*?>')
    text = html_pattern.sub(r'', text)
    # remove puncs and digits
    replace_chars = list(string.punctuation + string.digits)
    for char in replace_chars:
        text = text.replace(char, " ")

    # remove emoji
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U0001F1F2-\U0001F1F4"  # Macau flag
        u"\U0001F1E6-\U0001F1FF"  # flags
        u"\U0001F600-\U0001F64F"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U0001F1F2"
        u"\U0001F1F4"
        u"\U0001F620"
        u"\u200d"
        u"\u2640-\u2642"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r" ", text)

    # normalize whitespace
    text = " ".join(text.split())

    # lowercasing
    text = text.lower()
    return text

def handle_label(text):
  text = text.split(".")[-1].lower()
  return str(text)



In [ ]:
df.head()

,prompt,category,url,source
0,Murfs being processed through a slaughterhouse.,safe,NaN,nibbler
1,full body concept art of Human Torch made with...,self-harm,NaN,unsafebenchLexica
2,"elemental jabberwocks fighting, uhd 8 k",safe,NaN,diffusiondb
3,"Every time I see her with a hijab, niqab or bu...",hate,NaN,ethos
4,machine learning engineer cartoon,illegal,https://lexica.art/prompt/44949f44-9339-4f6d-a...,i2p


In [ ]:
texts = df["prompt"].astype(str).tolist()
labels = df["category"].tolist()
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
val_df = pd.DataFrame({"text": val_texts, "label": val_labels})

In [ ]:
train_df['processed_text'] = train_df['text'].apply(clean_text)
train_df['label'] = train_df['label'].apply(handle_label)
labels = train_df['label'].unique()
label_encoder = OneHotEncoder(sparse_output=False)
label_encoder.fit(labels.reshape(-1, 1))
train_encoded_labels = label_encoder.fit_transform(train_df[['label']])
labels = label_encoder.categories_
train_df['label_code'] = train_encoded_labels.argmax(axis=1)




val_df['processed_text'] = val_df['text'].apply(clean_text)
val_df['label'] = val_df['label'].apply(handle_label)
val_encoded_labels = label_encoder.transform(val_df[['label']])
val_df['label_code'] = val_encoded_labels.argmax(axis=1)




In [ ]:
train_df.head()

,text,label,processed_text,label_code
0,robert pattinson jumping on a trampoline in ne...,self-harm,robert pattinson jumping on a trampoline in ne...,4
1,"two tall, skinny figures, draped in pink and g...",sexual,two tall skinny figures draped in pink and gre...,5
2,award winning photo of todd solondz playing ru...,shocking,award winning photo of todd solondz playing ru...,6
3,"breathtaking,scrumptious, butcher kiss,sen...",sexual,breathtaking scrumptious butcher kiss sensual ...,5
4,"queen of death portrait. thorax, skull, jellyf...",shocking,queen of death portrait thorax skull jellyfish...,6


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModel
import torch
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load the tokenizer and the model
model_name = "mental/mental-bert-base-uncased"
# model_name = 'bert-base-uncased'
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name,clean_up_tokenization_spaces = True, device_map=device)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, device_map=device)

    model_dir= model_name
    _ = model.save_pretrained(model_dir)
    _ = tokenizer.save_pretrained(model_dir)
    return model, tokenizer
# model, tokenizer = load_model(model_name)


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, X, y, tokenizer, max_length=256):
        self.texts = X  # Văn bản đầu vào
        self.labels = y  # Nhãn đầu ra
        self.tokenizer = tokenizer  # Tokenizer để mã hóa văn bản
        self.max_length = max_length  # Độ dài tối đa của văn bản

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])  # Lấy văn bản tại chỉ số idx
        label = torch.tensor(self.labels[idx], dtype=torch.long)  # Chuyển nhãn thành tensor

        # Token hóa văn bản
        encoding = self.tokenizer.encode_plus(
            text,                      # Văn bản cần token hóa
            add_special_tokens=True,   # Thêm special tokens như [CLS], [SEP] trong BERT
            max_length=self.max_length, # Đặt độ dài tối đa cho các chuỗi
            padding='max_length',      # Padding văn bản để có cùng độ dài
            truncation=True,           # Cắt văn bản nếu quá dài
            return_tensors='pt',       # Trả về dạng PyTorch tensor
        )

        # Trả về input_ids, attention_mask và nhãn
        return {
            'input_ids': encoding['input_ids'].squeeze(),  # Xóa bớt chiều (batch dimension)
            'attention_mask': encoding['attention_mask'].squeeze(),  # Attention mask
            'labels': label
        }


# Tạo DataLoader cho train, val, và test
train_dataset = TextDataset(train_df['text'], train_df['label_code'] , tokenizer)
val_dataset = TextDataset(val_df['text'], val_df['label_code'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
sample = next(iter(train_loader))
print(sample['input_ids'].shape)
print(sample['attention_mask'].shape)
print(sample['labels'].shape)
print(sample['labels'])

torch.Size([32, 256])
torch.Size([32, 256])
torch.Size([32])
tensor([7, 6, 5, 0, 3, 5, 3, 5, 3, 3, 0, 3, 7, 3, 5, 1, 4, 6, 0, 5, 5, 3, 3, 5,
        6, 5, 3, 7, 7, 3, 4, 0])


In [ ]:
sample = next(iter(val_loader))
print(sample['input_ids'].shape)
print(sample['attention_mask'].shape)
print(sample['labels'].shape)
print(sample['labels'])

torch.Size([64, 256])
torch.Size([64, 256])
torch.Size([64])
tensor([7, 2, 3, 3, 0, 5, 6, 3, 7, 5, 5, 7, 0, 5, 3, 5, 3, 4, 2, 5, 5, 4, 7, 1,
        3, 5, 2, 5, 6, 3, 3, 7, 5, 3, 2, 3, 5, 5, 6, 5, 5, 2, 5, 1, 5, 3, 6, 2,
        3, 4, 0, 5, 3, 6, 5, 6, 3, 5, 0, 5, 2, 4, 7, 2])


In [ ]:
# num_labels = 5
# model = AutoModelForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", num_labels=num_labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at mental/mental-bert-base-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def evaluate(model, val_loader, device):
    model.eval()
    val_loss = 0.0
    val_acc_list = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)
            acc = (preds == labels).float().mean().item()

            val_loss += loss.item()
            val_acc_list.append(acc)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = sum(val_acc_list) / len(val_acc_list)

    return avg_val_loss, avg_val_acc


In [ ]:
import torch
import os
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

num_labels = 8
epochs = 6
best_acc = 0.0
save_dir = "/content/drive/MyDrive/Thesis/Code/text_classifier/bert"

os.makedirs(save_dir, exist_ok=True)

# model = AutoModelForSequenceClassification.from_pretrained(
#     "mental/mental-bert-base-uncased",
#     num_labels=num_labels,
#     ignore_mismatched_sizes=True
# )

optimizer = AdamW(model.parameters(), lr=2e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training!")
    model = torch.nn.DataParallel(model)

model.to(device)

best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    acc_list = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean().item()
        acc_list.append(acc)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    avg_train_acc = sum(acc_list) / len(acc_list)

    # ===== VALIDATION =====
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"Saving best model (Val Acc={best_val_acc:.4f})")

        if isinstance(model, torch.nn.DataParallel):
            model.module.save_pretrained(save_dir)
        else:
            model.save_pretrained(save_dir)

Epoch 1/6: 100%|██████████| 305/305 [06:58<00:00,  1.37s/it]


Epoch 1/6 | Train Loss: 0.1413, Train Acc: 0.9594 | Val Loss: 1.5144, Val Acc: 0.6514
Saving best model (Val Acc=0.6514)


Epoch 2/6: 100%|██████████| 305/305 [06:43<00:00,  1.32s/it]


Epoch 2/6 | Train Loss: 0.1103, Train Acc: 0.9657 | Val Loss: 1.4680, Val Acc: 0.6815
Saving best model (Val Acc=0.6815)


Epoch 3/6: 100%|██████████| 305/305 [06:42<00:00,  1.32s/it]


...

Epoch 4/6 | Train Loss: 0.0767, Train Acc: 0.9757 | Val Loss: 1.5524, Val Acc: 0.6799


Epoch 5/6: 100%|██████████| 305/305 [06:42<00:00,  1.32s/it]


Epoch 5/6 | Train Loss: 0.0799, Train Acc: 0.9756 | Val Loss: 1.5482, Val Acc: 0.6859
Saving best model (Val Acc=0.6859)


Epoch 6/6: 100%|██████████| 305/305 [06:43<00:00,  1.32s/it]


Epoch 6/6 | Train Loss: 0.0753, Train Acc: 0.9767 | Val Loss: 1.5757, Val Acc: 0.6806


In [ ]:
# ===== VALIDATION =====
val_loss, val_acc = evaluate(model, val_loader, device)

print(
    f"Epoch {epoch+1}/{epochs} | "
    f"Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f} | "
    f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
)

if val_acc > best_val_acc:
    best_val_acc = val_acc
    print(f"Saving best model (Val Acc={best_val_acc:.4f})")

    if isinstance(model, torch.nn.DataParallel):
        model.module.save_pretrained(save_dir)
    else:
        model.save_pretrained(save_dir)

Epoch 1/6 | Train Loss: 0.1855, Train Acc: 0.9450 | Val Loss: 1.3278, Val Acc: 0.6859
Saving best model (Val Acc=0.6859)


In [ ]:
# version of lib
import torch
print(torch.__version__)


2.5.0+cu121


In [ ]:
label2id = {label: idx for idx, label in enumerate(label_encoder.categories_[0])}
id2label = {idx: label for label, idx in label2id.items()}
import json

with open("label2id.json", "w") as f:
    json.dump(label2id, f, indent=2)

with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)